# Model Validation Analysis

Human-annotated evaluation of 46 questions across 6 LLM × embedding combinations.

- **T** = Correct (1.0)
- **PC** = Partially Correct (0.5)
- **F** = Wrong (0.0)

Difficulty tiers: Easy (Q1–Q17), Medium (Q18–Q31), Hard (Q32–Q46)

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

df = pd.read_csv('../../Model Validation - Sheet1.csv', index_col=0)
df = df.drop(columns=['Question', 'Notes'], errors='ignore')
df.index = [f'Q{i+1}' for i in range(len(df))]

MODELS = [
    'Claude + Ollama Embeddings',
    'Claude + Sentence Embeddings',
    'Claude + OpenAI Embeddings',
    'Ollama + Ollama Embeddings',
    'Ollama + Sentence Embeddings',
    'Ollama + OpenAI Embeddings',
]
df.columns = MODELS

SCORE_MAP = {'T': 1.0, 'PC': 0.5, 'F': 0.0}
scores = df.applymap(lambda x: SCORE_MAP.get(str(x).strip(), np.nan))

EASY   = [f'Q{i}' for i in range(1, 18)]
MEDIUM = [f'Q{i}' for i in range(18, 32)]
HARD   = [f'Q{i}' for i in range(32, 47)]

print('Data loaded:', scores.shape)

Data loaded: (46, 6)


C:\Users\Juan\AppData\Local\Temp\ipykernel_50548\2564969532.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  scores = df.applymap(lambda x: SCORE_MAP.get(str(x).strip(), np.nan))


## Overall Accuracy per Model

In [ ]:
overall = scores.mean().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(5, 3.2))
colors = ['#0072B2' if 'Claude' in m else '#E69F00' for m in overall.index]
bars = ax.barh(range(len(overall)), overall.values, color=colors,
               edgecolor='black', linewidth=0.5, height=0.55)

for bar, val in zip(bars, overall.values):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9, fontweight='bold')

short_labels = [m.replace(' Embeddings', '').replace(' + ', ' + ') for m in overall.index]
ax.set_yticks(range(len(overall)))
ax.set_yticklabels(short_labels, fontsize=9)
ax.set_xlabel('Mean Score (T=1, PC=0.5, F=0)', fontsize=9)
ax.set_xlim(0, 1.1)
ax.set_title('Overall Accuracy by Model Combination', fontsize=10, pad=8)
ax.axvline(0.75, color='gray', linestyle='--', linewidth=0.8)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#0072B2', label='Claude'),
                   Patch(facecolor='#E69F00', label='Ollama'),
                   plt.Line2D([0], [0], color='gray', linestyle='--', label='0.75')]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('overall_accuracy.png', dpi=200, bbox_inches='tight')
plt.show()
print(overall.sort_values(ascending=False).round(3).to_string())

## Accuracy by Difficulty Tier

In [ ]:
tier_scores = pd.DataFrame({
    'Easy':   scores.loc[EASY].mean(),
    'Medium': scores.loc[MEDIUM].mean(),
    'Hard':   scores.loc[HARD].mean(),
})

short = [m.replace(' Embeddings', '').replace(' + ', '\n') for m in tier_scores.index]

fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(len(tier_scores))
w = 0.22
for i, (tier, color) in enumerate(zip(['Easy', 'Medium', 'Hard'], ['#009E73', '#56B4E9', '#D55E00'])):
    bars = ax.bar(x + i*w, tier_scores[tier], w, label=tier, color=color,
                  edgecolor='black', linewidth=0.5)
    for bar, val in zip(bars, tier_scores[tier]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x + w)
ax.set_xticklabels(short, fontsize=8)
ax.set_ylabel('Mean Score', fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_title('Accuracy by Difficulty Tier', fontsize=10)
ax.legend(fontsize=8, framealpha=0.7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('accuracy_by_tier.png', dpi=200, bbox_inches='tight')
plt.show()
print(tier_scores.round(3).to_string())

## T / PC / F Breakdown per Model

In [ ]:
counts = {}
for col in MODELS:
    vc = df[col].str.strip().value_counts()
    counts[col] = {'T': vc.get('T', 0), 'PC': vc.get('PC', 0), 'F': vc.get('F', 0)}
counts_df = pd.DataFrame(counts).T

short = [m.replace(' Embeddings', '').replace(' + ', '\n') for m in counts_df.index]

fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.arange(len(counts_df))
bottom_t = np.zeros(len(counts_df))
bottom_pc = counts_df['T'].values.astype(float)
bottom_f  = bottom_pc + counts_df['PC'].values.astype(float)

ax.bar(x, counts_df['T'],  color='#009E73', edgecolor='black', linewidth=0.4, label='Correct (T)')
ax.bar(x, counts_df['PC'], bottom=bottom_pc, color='#F0E442', edgecolor='black', linewidth=0.4, label='Partial (PC)')
ax.bar(x, counts_df['F'],  bottom=bottom_f,  color='#D55E00', edgecolor='black', linewidth=0.4, label='Wrong (F)')

ax.set_xticks(x)
ax.set_xticklabels(short, fontsize=8)
ax.set_ylabel('Count', fontsize=9)
ax.set_title('Correct / Partial / Wrong per Model', fontsize=10)
ax.legend(fontsize=8, loc='lower right', framealpha=0.7)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('tpcf_counts.png', dpi=200, bbox_inches='tight')
plt.show()
print(counts_df.to_string())

## Heatmap: Score per Question per Model

In [ ]:
fig, ax = plt.subplots(figsize=(7, 11))
cmap = mcolors.LinearSegmentedColormap.from_list('cb_tfpc', ['#D55E00', '#F0E442', '#009E73'])
im = ax.imshow(scores.values, aspect='auto', cmap=cmap, vmin=0, vmax=1)

short_cols = [m.replace(' Embeddings', '').replace(' + ', '\n') for m in MODELS]
ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels(short_cols, fontsize=9)
ax.set_yticks(range(len(scores)))
ax.set_yticklabels(scores.index, fontsize=8)

# Tier separator lines
for sep in [16.5, 30.5]:
    ax.axhline(sep, color='black', linewidth=2)

# Cell labels
for i in range(len(scores)):
    for j in range(len(MODELS)):
        val = scores.values[i, j]
        label = {1.0: 'T', 0.5: 'PC', 0.0: 'F'}.get(val, '')
        ax.text(j, i, label, ha='center', va='center', fontsize=7,
                color='white' if val < 0.3 else 'black')

# Vertical tier labels on the right
tier_info = [('Easy', 0, 16), ('Medium', 17, 30), ('Hard', 31, 45)]
n_cols = len(MODELS)
for name, start, end in tier_info:
    mid = (start + end) / 2
    ax.text(n_cols + 0.1, mid, name, ha='left', va='center', fontsize=9,
            fontweight='bold', rotation=90, transform=ax.transData)

ax.set_xlim(-0.5, n_cols + 0.6)

plt.colorbar(im, ax=ax, label='Score', shrink=0.4)
ax.set_title('Per-Question Scores by Model', fontsize=11, pad=10)
plt.tight_layout()
plt.savefig('heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

## Claude vs Ollama Summary

In [12]:
claude_cols = [m for m in MODELS if 'Claude' in m]
ollama_cols = [m for m in MODELS if 'Ollama' in m and 'Claude' not in m]

claude_mean = scores[claude_cols].values.mean()
ollama_mean = scores[ollama_cols].values.mean()

print(f'Claude average (all embeddings): {claude_mean:.3f}')
print(f'Ollama average (all embeddings): {ollama_mean:.3f}')
print()
print('Per embedding comparison:')
for emb in ['Ollama Embeddings', 'Sentence Embeddings', 'OpenAI Embeddings']:
    c = scores[f'Claude + {emb}'].mean()
    o = scores[f'Ollama + {emb}'].mean()
    print(f'  {emb:<25}  Claude={c:.3f}  Ollama={o:.3f}  diff={c-o:+.3f}')

Claude average (all embeddings): 0.891
Ollama average (all embeddings): 0.601

Per embedding comparison:
  Ollama Embeddings          Claude=0.848  Ollama=0.630  diff=+0.217
  Sentence Embeddings        Claude=0.935  Ollama=0.565  diff=+0.370
  OpenAI Embeddings          Claude=0.891  Ollama=0.609  diff=+0.283
